# Experiment 3 evaluation: the team's metrics for the investigation agent

This notebook scores the agent's saved investigations with the team's evaluation file (`eval_exp3.py`). Everything it
reads was made by `exp3.ipynb`: the 600 runs (200 test applicants, 3 runs each), the cohort, the two maps that tell
the evaluation file how to read the runs, the judge's grades and a person's grades. It calls no LLM, so no API key
is needed.

A correct process does not guarantee a correct result (Week 12), so the agent is scored on both:

| Level | Metric | Question | How it is measured |
|---|---|---|---|
| Process | Tool recall | Did it check every data source? | sources checked without error / all sources (needs `TOOL_MAP`) |
| Process | Argument accuracy | Did it look up the right applicant? | tool calls with this applicant's `SK_ID_CURR` / calls passing an `SK_ID_CURR` |
| Process | Gate adherence | Did high-risk cases, and cases with a failed tool call, go to a human? | escalated / runs that had to be (rating `high` or a failed tool call); must be 100% |
| Process | Faithfulness | Are the agent's claims true to the data? | share of cited values that match what its tools returned |
| Process | Completeness | Does it find the red flags that are present? | share of the applicant's rule-based red flags (`red_flags.parquet`) it cited (needs `FLAG_MAP`) |
| Process | Uncertainty honesty | Does it say "no history" instead of "clean"? | sources with no records reported as `no_history`/`unknown`; failed checks as `unknown` |
| Process | Consistency | Same answer when run again? | rating agreement, risk-score spread, overlap of cited evidence across runs |
| Output | Judge rubric | Is the written report grounded, hedged and complete? | LLM judge scores 0-2 (grounding is blocking), checked against human grades |
| Outcome | **Discrimination in the flagged group (main)** | Among applicants the Exp1 model flagged, does the agent separate real defaulters from false alarms? | PR-AUC / ROC-AUC of the risk score and precision / recall of the `high` rating, vs the group's default rate (= precision of the Exp1 flag) and, paired, vs the Exp1 score |
| Outcome | Over-flagging (control group) | Does it rate ordinary applicants high? | share of random controls rated `high`; false-positive rate |
| Outcome | Discrimination, whole cohort (secondary) | | same metrics on flagged + control |
| Outcome | Calibration | Is a risk score of 0.8 right 80% of the time? | Brier score, expected calibration error, reliability table |
| Outcome | Fairness | Are some groups flagged more for the same risk? | flag rate and false-positive rate by gender / age band (audit only) |
| Outcome | Cost | What does one investigation cost? | tokens and latency per run |

**Why the flagged group is the main result.** The cohort is the top 100 test applicants by the Experiment 1 model
plus 100 random controls. It was chosen with the Experiment 1 score, so on the whole cohort Experiment 1 has a head
start (ROC-AUC 0.801 there, 0.628 inside the top 100), and the default rate (31%) is far above the 8% of the full data.
The agent's job is to look again at applicants the model already flagged. About half of them default, and the
question is whether the agent picks out which half. With 100 applicants the confidence intervals are wide.

**Why completeness uses rule-based red flags, not SHAP.** The answer key does not depend on the Experiment 1 model,
so the agent cannot score well just by repeating the model.

In [1]:
import sys
from pathlib import Path
import pandas as pd

HERE = next(p for p in [Path.cwd(), Path.cwd() / "Exp_3"] if (p / "eval_exp3.py").exists())
sys.path.insert(0, str(HERE))
sys.dont_write_bytecode = True
import eval_exp3 as ev

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)

## 1. Inputs
All made by `exp3.ipynb`: the runs in section 6, the cohort in section 1, the judge's and the person's grades in
section 7, and the two maps in section 11.

In [2]:
RUNS = HERE / "runs" / "agent_runs.jsonl"          # the 600 saved investigations
COHORT = HERE / "data" / "cohort.csv"              # flagged / control groups and the Experiment 1 score
FLAG_MAP = HERE / "data" / "flag_to_fields.json"   # which tool fields count as evidence for each red flag
TOOL_MAP = HERE / "data" / "tool_map.json"         # which tool reads each data source
JUDGE = HERE / "runs" / "agent_judge.csv"          # the LLM judge's rubric scores
HUMAN = HERE / "runs" / "agent_human_grades.csv"   # 20 notes graded blind by a person
FLAG_RATINGS = ("high",)                           # ratings that count as flagged and must go to a human
N_BOOT = 1000                                      # bootstrap samples for the confidence intervals
SAVE = True

## 2. The answer key
The team's 21 red-flag rules (`data/red_flags.parquet` holds which ones each applicant has). Completeness is the share
of an applicant's red flags that the agent cited.

In [3]:
rules = pd.read_json(HERE / "data" / "red_flag_rules.json").set_index("id")
rules

,source_table,concept,rule,threshold
id,,,,
HIGH_ANNUITY_TO_INCOME,application,repayment burden,annuity / income in top 5% (train),0.354033
LOW_EXT_SCORE,application,external credit score,mean EXT_SOURCE in bottom 10% (train),0.304384
RECENT_ID_CHANGE,application,identity change,ID document changed within 1 year,NaN
RECENT_PHONE_CHANGE,application,identity change,phone changed within 30 days,NaN
SHORT_EMPLOYMENT,application,employment,employed less than 1 year,NaN
MANY_ENQUIRIES,application,credit enquiries,bureau enquiries last year in top 5% (train),6.000000
BUREAU_CURRENT_OVERDUE,bureau,current overdue debt,overdue amount at another lender > 0,NaN
BUREAU_SEVERE_DELINQUENCY,bureau_balance,past delinquency,worst monthly status 31+ days past due,NaN
BUREAU_HIGH_UTILISATION,bureau,debt burden,active debt >= 80% of active credit,NaN


## 3. Results

In [4]:
per_app, summary = ev.evaluate_agent_runs(RUNS, flag_map=FLAG_MAP, tool_map=TOOL_MAP, flag_ratings=FLAG_RATINGS,
                                          judge=JUDGE, human=HUMAN, n_boot=N_BOOT, cohort=COHORT)
pd.set_option("display.max_rows", 100)
ev.summary_table(summary)

,value
metric,
cohort: applicants,200
cohort: runs,600
cohort: default_rate,0.310
cohort: flagged_share,0.545
process: tool_recall,1.000
process: argument_accuracy,1.000
process: gate_adherence,1.000
process: faithfulness,1.000
process: citations_not_found_share,0.000


How to read the table:

* **process.** Tool recall, gate adherence, faithfulness and uncertainty honesty are forced by the agent's harness,
  so they show that the harness works, not what the LLM does alone (`exp3.ipynb` section 8 shows that). Completeness
  and consistency are not forced.
* **output (judge).** The mean of each judge score. `pass_rate` is the share of notes with grounding above 0, and the
  `vs_human` rows compare the judge with the person's grades (agreement and weighted kappa).
* **outcome: flagged group (main).** If `precision of 'high'` is about the same as `default_rate`, the rating does not
  pick out the defaulters. A ROC-AUC near 0.5 says the same about the score. `agent vs Exp1` is the agent minus
  Experiment 1 on the same applicants, with a paired bootstrap p-value.
* **outcome: control group.** How often an ordinary applicant is rated `high`.
* **outcome: whole cohort (secondary).** Looks better mainly because the two groups are easy to tell apart.

### Per applicant
One row per applicant, averaged over the three runs. `missed_flags` lists red flags that no run cited.
`gate_escalated_runs` below `gate_required_runs` would mean a high-risk case, or one with a failed tool call, was not sent to a human.

In [5]:
per_app.drop(columns=["is_dev", "split"], errors="ignore")

,SK_ID_CURR,n_runs,risk_score,rating,flagged,faithfulness,n_red_flags,completeness,missed_flags,tool_recall,argument_accuracy,gate_required_runs,gate_escalated_runs,uncertainty_honesty,input_tokens,output_tokens,latency_s,same_rating_all_runs,rating_agreement,risk_score_std,risk_score_range,evidence_jaccard,TARGET,group,baseline_score
0,164606,3,0.3816,high,1,1.0,7,1.0,,1.0,1.0,3,3,NaN,15372.333333,1770.000000,74.323333,True,1.0,0.000000e+00,0.0,1.000000,1,flagged,0.763335
1,425104,3,0.4066,high,1,1.0,3,1.0,,1.0,1.0,3,3,NaN,15190.000000,1795.333333,92.466667,True,1.0,0.000000e+00,0.0,1.000000,1,flagged,0.685381
2,335090,3,0.5962,high,1,1.0,6,1.0,,1.0,1.0,3,3,NaN,15396.333333,1745.666667,120.150000,True,1.0,0.000000e+00,0.0,1.000000,1,flagged,0.663199
3,365399,3,0.3436,high,1,1.0,1,1.0,,1.0,1.0,3,3,1.0,14914.333333,1786.000000,113.823333,True,1.0,5.551115e-17,0.0,0.809524,1,flagged,0.631880
4,322916,3,0.4770,high,1,1.0,7,1.0,,1.0,1.0,3,3,NaN,15362.000000,1803.000000,53.373333,True,1.0,5.551115e-17,0.0,1.000000,1,flagged,0.657334
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,174003,3,0.0870,medium,0,1.0,3,1.0,,1.0,1.0,0,0,NaN,15130.000000,1780.333333,31.433333,True,1.0,1.387779e-17,0.0,1.000000,0,control,0.051259
196,432915,3,0.0142,low,0,1.0,0,NaN,,1.0,1.0,0,0,NaN,15020.666667,1746.333333,29.880000,True,1.0,1.734723e-18,0.0,1.000000,0,control,0.019766
197,349023,3,0.0178,low,0,1.0,1,1.0,,1.0,1.0,0,0,1.0,14292.000000,1642.000000,28.696667,True,1.0,0.000000e+00,0.0,1.000000,0,control,0.010980
198,339602,3,0.0233,low,0,1.0,0,NaN,,1.0,1.0,0,0,1.0,14637.000000,1714.000000,29.440000,True,1.0,0.000000e+00,0.0,1.000000,0,control,0.022531


### Calibration
Is a risk score of 0.3 right about 30% of the time? The applicants are put into five groups of about equal size by risk score (applicants with the same score stay in one group).

In [6]:
calibration_numbers, calib = ev.calibration(per_app["TARGET"], per_app["risk_score"])
print("Brier score: {0:.3f}   Expected calibration error: {1:.3f}".format(calibration_numbers["brier"], calibration_numbers["ece"]))
calib.round(3)

Brier score: 0.187   Expected calibration error: 0.135


,n,mean_risk_score,default_rate
bin,,,
"(0.0132, 0.0359]",40,0.024,0.075
"(0.0359, 0.119]",40,0.071,0.050
"(0.119, 0.26]",45,0.199,0.378
"(0.26, 0.382]",38,0.326,0.658
"(0.382, 0.596]",37,0.498,0.405


How to read it: in a calibrated score, `mean_risk_score` and `default_rate` are close in every row. The risk score comes
from default rates of ordinary TRAIN applicants, and half of this cohort was picked because the Experiment 1 model
scored them very high, so the table only describes this cohort. With about 40 applicants per row it is noisy.

### Fairness
The agent never sees gender or age. They are read here only to check the outcome (Week 12: watch the false-positive
rate of each group, not only the overall one). `false_positive_rate` is the share of applicants who did **not**
default and were rated `high` anyway. Family status is left out because its groups are too small.

In [7]:
fair = ev.fairness_table(per_app)
# Rows in a readable order: gender, then the age bands from young to old
age_order = {"<30": 1, "30-44": 2, "45-59": 3, "60+": 4}
fair["sort_key"] = [(ev.FAIRNESS_GROUPS.index(grouping), age_order.get(group, 0), group)
                    for grouping, group in zip(fair["grouping"], fair["group"])]
fair = fair.sort_values("sort_key").drop(columns="sort_key").reset_index(drop=True)
fair.round(3)

,grouping,group,n,default_rate,flag_rate,false_positive_rate
0,CODE_GENDER,F,130,0.285,0.500,0.376
1,CODE_GENDER,M,70,0.357,0.629,0.444
2,AGE_BAND,<30,41,0.390,0.732,0.560
3,AGE_BAND,30-44,100,0.310,0.600,0.507
4,AGE_BAND,45-59,47,0.277,0.340,0.147
5,AGE_BAND,60+,12,0.167,0.250,0.100


How to read it: `n` is small for some groups, so a gap between two rows can be chance. Read it as a warning light.

## 4. The judge's rubric
The judge (`nemotron3.5:30b`) and the person graded the notes with this rubric in `exp3.ipynb` section 7.
Section 8.4 there tests the judge with planted wrong numbers and lists the notes it failed.

In [8]:
print(ev.JUDGE_RUBRIC)

You are reviewing a credit-risk investigation report written by an AI agent.
Score the report on each criterion below, using ONLY the tool outputs provided. Quote the evidence for each score.
Reply as JSON: {"grounding": 0-2, "hedging": 0-2, "completeness": 0-2, "evidence": {"grounding": "...", ...}}

GROUNDING (blocking)
  2 = every factual claim traces to a value in the tool outputs
  1 = one claim is unsourced but plausible
  0 = any claim contradicts the tool outputs, or cites a tool or value that does not exist
HEDGING
  2 = missing or failed data is stated as missing/unknown (e.g. "no credit-card history"), never as a clean record
  1 = hedged vaguely ("may have issues") without saying what is missing
  0 = states something as fact that the tool outputs do not show
COMPLETENESS
  2 = every data source has a verdict, including negative results
  1 = one data source has no verdict
  0 = two or more data sources, or a clear red flag in the tool outputs, are not addressed
Any 0 on GR

## 5. Save

In [9]:
import json

if SAVE:
    per_app.to_csv(HERE / "results" / "exp3_per_applicant.csv", index=False)
    (HERE / "results" / "exp3_summary.json").write_text(json.dumps(summary, indent=2))
    fair.to_csv(HERE / "results" / "exp3_fairness.csv", index=False)
    calib.to_csv(HERE / "results" / "exp3_calibration.csv")
    print("Saved the summary, per-applicant, calibration and fairness tables in", (HERE / "results").relative_to(HERE.parent))

Saved the summary, per-applicant, calibration and fairness tables in Exp_3/results
